In [ ]:
import sys
sys.path.append("../")
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle 
import pandas as pd

# import data
all_df = pd.read_csv("../data/mimic-hallu.csv")
# metric_df = pd.DataFrame()
metric_df = pd.read_csv("../en-metric.csv")

cols_to_lower = ['generation', 'human_reference', 'patient_file']
all_df[cols_to_lower] = all_df[cols_to_lower].apply(lambda x: x.str.lower())

## BLEU 
[SacreBLEU](https://github.com/mjpost/sacrebleu) 

In [ ]:
import numpy as np
from sacrebleu.metrics import BLEU, CHRF
from nltk.tokenize import sent_tokenize
import re
from tqdm import tqdm
tqdm.pandas()

In [ ]:
from sacrebleu.metrics import BLEU

bleu = BLEU()  

def compute_bleu_row(row):
    if pd.isna(row['generation']) or pd.isna(row['human_reference']):
        return {
            'bleu_1': None,
            'bleu_1gram_precision_1': None,
            'bleu_2gram_precision_1': None,
            'bleu_3gram_precision_1': None,
            'bleu_4gram_precision_1': None,
        }
    
    hypothesis = row['generation']      # full document string
    reference  = row['human_reference']      

    result = bleu.corpus_score(
        [hypothesis],      # list of 1 document
        [[reference]]      # list of list of 1 reference
    )
    
    return {
        'bleu_1': result.score,
        'bleu_1gram_precision_1': result.precisions[0],
        'bleu_2gram_precision_1': result.precisions[1],
        'bleu_3gram_precision_1': result.precisions[2],
        'bleu_4gram_precision_1': result.precisions[3],
    }
    
bleu_scores = all_df.progress_apply(compute_bleu_row, axis=1, result_type='expand')

In [ ]:
metric_df = pd.concat([metric_df, bleu_scores], axis=1)

In [ ]:
metric_df = metric_df.drop(['bleu_4gram_precision'], axis=1)

## SelfBLEU


1. Choose one sentence and calculate the BLEU score between the sentence and remain sentenses.
2. Change the sentence and return to step-1 untill there are no sentence you can choose.
3. take the average of the scores calculated in step1-2


In [ ]:
import numpy as np
import copy
from sacrebleu.metrics import BLEU

def calculate_selfbleu(row): 
    score_list = []
    sent_list = sent_tokenize(row['generation'], language='english')  #all sentences
    num = len(sent_list) #number of sentences
    for i in range(num):
        refs = [[sent_list[j]] for j in range(num) if j!=i]
        sys = [sent_list[i]]  
        blue = bleu.corpus_score(sys, refs) #calcuate score
        score_list.append(blue.score) #save the score
        #show each score (Comment out if not needed)
        return sum(score_list) / num

bleu = BLEU()
selfbleu_scores = all_df.progress_apply(calculate_selfbleu, axis=1)


In [ ]:
metric_df['self_bleu'] = selfbleu_scores

## ChrF

In [ ]:
from sacrebleu.metrics import CHRF

chrfp  = CHRF(word_order=1)  # ChrF+
chrfpp = CHRF(word_order=2)  # ChrF++

def compute_chrf_row(row):
    if pd.isna(row['generation']) or pd.isna(row['human_reference']):
        return {'chrf+': None, 'chrf++': None}

    hypothesis = row['generation']   # full document string
    reference  = row['human_reference']   

    chrfp_result  = chrfp.corpus_score(
        [hypothesis],
        [[reference]]
    )
    chrfpp_result = chrfpp.corpus_score(
        [hypothesis],
        [[reference]]
    )

    return {
        'chrf+_1':  chrfp_result.score,
        'chrf++_1': chrfpp_result.score
    }

chrf_scores = all_df.progress_apply(compute_chrf_row, axis=1, result_type='expand')

In [ ]:
metric_df = pd.concat([metric_df, chrf_scores], axis=1)

## MoverScore
MoverScore (Zhao et.al, 2019) is a monolingual measure of evaluating the similarity between a sentence pair written in the same language.
code adapted from: https://github.com/AIPHES/emnlp19-moverscore/tree/master

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from metric_utils_en import moverscore_corpus
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
def compute_moverscore_row(row):
    if pd.isna(row['generation']) or pd.isna(row['human_reference']):
        return {
            'mover_score': None,
        }
    
    generation_sents = sent_tokenize(row['generation'], language='english')  
    reference_sents = sent_tokenize(row['human_reference'], language='english')
    
    # mover score requires generation has the same sentence length as system
    # we pad the shorter ones with empty string '' as extra sentences 
    # Compare lengths
    len_gen = len(generation_sents)
    len_ref = len(reference_sents)

    if len_gen < len_ref:
        # Pad generation_sents
        generation_sents += [''] * (len_ref - len_gen)
    elif len_ref < len_gen:
        # Pad reference_sents
        reference_sents += [''] * (len_gen - len_ref)
    
    result = moverscore_corpus(
        sys_stream=generation_sents,
        ref_streams=[reference_sents]
    )
    
    return {
        'mover_score': result,
    }

mover_scores = all_df.progress_apply(compute_moverscore_row, axis=1, result_type='expand')


In [ ]:
metric_df = pd.concat([metric_df, mover_scores], axis=1)

## Word Mover's Distance 

WMD enables us to assess the “distance” between two documents in a meaningful way even when they have no words in common. The intuition behind the method is that we find the minimum “traveling distance” between documents, in other words the most efficient way to “move” the distribution of document 1 to the distribution of document 2. 

We use fasttext embeddings in Dutch to create static word embeddings. https://fasttext.cc/docs/en/crawl-vectors.html

Note: BERT embeddings is not suitable for this task because it is a contextual embedding. There are two issues with using WMD on BERT embeddings (https://stackoverflow.com/questions/72506033/bert-with-wmd-distance-for-sentence-similarity):


- BERT embeddings provide contextual representation of sub-words and the sentence (representation of of a subword changes in different context).
- There is no measure of density or weight on words and sub-words other than the attention mask on tokens.



In [ ]:
from gensim.models import KeyedVectors
from metric_utils_en import preprocessing

# Load English fastText embeddings
fasttext_model = KeyedVectors.load_word2vec_format("../models/cc.en.300.vec")

def compute_wmd(row):
    wmd_score = fasttext_model.wmdistance(preprocessing(row['generation']), preprocessing(row['human_reference']))
    
    return {
    'WMD': wmd_score}

In [ ]:
metric_df['WMD'] = all_df.progress_apply(compute_wmd, axis=1, result_type='expand')

## Sentence Mover Similarity (SMS)

https://github.com/eaclark07/sms/blob/master/wmd-relax-master/smd.py#L90


In [ ]:
# Ensure you have the large Dutch spacy model with word vectors
# In your terminal, run: python -m spacy download nl_core_web_lg
from smd_en import calculate_smd

def compute_sms_row(row):
    if pd.isna(row['generation']) or pd.isna(row['human_reference']):
        return None
    
    score = calculate_smd(
        reference_text=row['human_reference'],
        hypothesis_text=row['generation'],
        metric="sms" # Explicitly using Sentence Mover's Similarity
    )
    return score

# Apply the function to each row
sms_scores = all_df.progress_apply(compute_sms_row, axis=1)

# Add the scores to your metrics DataFrame
# metric_df['sms'] = sms_scores

In [ ]:
metric_df['sms'] = sms_scores

## WEEM4TS

In [ ]:
from metric_utils_en import weem4ts

# Load Dutch fastText embeddings
model = KeyedVectors.load_word2vec_format("../models/cc.en.300.vec")
embwords = list(model.key_to_index)

In [ ]:
metric_df['weem4ts'] = all_df.progress_apply(
    lambda row: weem4ts(row['human_reference'], row['generation'], model, embwords, lang='en'),
    axis=1
)

## BERTScore 
https://github.com/Tiiiger/bert_score/blob/master/example/Demo.ipynb


In [ ]:
from bert_score import score

In [ ]:
import os
print(os.path.exists("../models/Bio_ClinicalBERT"))

In [ ]:
from transformers import AutoTokenizer

path = "../models/Bio_ClinicalBERT"

tok = AutoTokenizer.from_pretrained(path, local_files_only=True)
print("tokenizer ok")

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("../models/Bio_ClinicalBERT")
print(model.config.num_hidden_layers)

In [ ]:
ref = all_df['human_reference'].to_list()
preds = all_df['generation'].to_list()

P, R, F1 = score(preds, ref, lang='en', model_type="../models/Bio_ClinicalBERT", num_layers=12, rescale_with_baseline=True)

bert_scores = pd.DataFrame ({
    'bertscore_precision': P.tolist(),
    'bertscore_recall': R.tolist(),
    'bertscore_f1': F1.tolist()})

In [ ]:

ref = all_df['human_reference'].to_list()
preds = all_df['generation'].to_list()

P, R, F1 = score(preds, ref, lang='en', model_type='FacebookAI/roberta-large-mnli', num_layers=19, rescale_with_baseline=True)

bert_scores = pd.DataFrame ({
    'bertscore_precision': P.tolist(),
    'bertscore_recall': R.tolist(),
    'bertscore_f1': F1.tolist()})

In [ ]:
metric_df = pd.concat([metric_df, bert_scores], axis=1)

## Sentence BERT

In [ ]:
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize
import numpy as np
from tqdm.auto import tqdm

model = SentenceTransformer('emilyalsentzer/Bio_ClinicalBERT')

def get_doc_embedding(text, model): 
    sentences = sent_tokenize(text, language='english')  
    sent_embeddings = model.encode(sentences)
    
    return np.mean(sent_embeddings, axis=0)


def calculate_sent_bert(row): 
    ref_emb = get_doc_embedding(row["human_reference"], model)
    gen_emb = get_doc_embedding(row["generation"], model)   
    
    similarity = util.cos_sim(ref_emb, gen_emb)
    
    return similarity.item()

tqdm.pandas()
sent_bert_scores = all_df.progress_apply(calculate_sent_bert, axis=1)


In [ ]:
metric_df["sent_bert"] = sent_bert_scores

## ROUGE1, ROUGE2, ROUGE-L, ROUGE-S, ROUGE-SU, ROUGE-W
https://pypi.org/project/rouge-metric/


In [ ]:
from rouge_metric import PyRouge
from metric_utils_en import preprocessing

def compute_rouges_su_w_row(row):
    
    # preprocess the string: lemmatization, remove punactuation and stopwords, remove tags like [persoon-1], [ziekenhuis-1], etc
    hypothesis = [preprocessing(row['generation'])]
    reference = [[preprocessing(row['human_reference'])]]

    rouge = PyRouge(rouge_n=(1, 2, 4), rouge_l=True, rouge_w=True,
                    rouge_w_weight=1.2, rouge_s=True, rouge_su=True, skip_gap=4)
    scores = rouge.evaluate(hypothesis, reference)
    

    return {
        'rouge1_f1': scores['rouge-1']['f'],
        'rouge2_f1': scores['rouge-2']['f'],
        'rougeL_f1': scores['rouge-l']['f'],
        'rouge-w-1.2': scores['rouge-w-1.2']['f'], 
        'rouge-s4': scores['rouge-s4']['f'], 
        'rouge-su4': scores['rouge-su4']['f']
    }


In [ ]:
# Apply to DataFrame
rouge_scores = all_df.progress_apply(compute_rouges_su_w_row, axis=1, result_type='expand')

In [ ]:
metric_df = pd.concat([metric_df, rouge_scores], axis=1)

## METEOR
https://huggingface.co/spaces/evaluate-metric/meteor


In [ ]:
from metric_utils_en import meteor 

In [ ]:
# Load METEOR metric
# meteor = evaluate.load('meteor')

# Function to compute METEOR per row
def compute_meteor_row(row):
    if pd.isna(row['generation']) or pd.isna(row['human_reference']):
        return {
            'meteor_score': None
        }

    result = meteor(
        predictions = [row['generation']],
        references = [[row['human_reference']]]
    )

    return {
        'meteor': result['meteor']
    }

# Apply to each row
meteor_scores = all_df.progress_apply(compute_meteor_row, axis=1, result_type='expand')

# # Concatenate with original DataFrame
# metrics_per_letter_df = pd.concat([metrics_per_letter_df, meteor_scores], axis=1)

In [ ]:
metric_df['meteor'] = meteor_scores

## NIST
https://www.nltk.org/api/nltk.translate.nist_score.html

In [ ]:
from nltk.translate.nist_score import sentence_nist, corpus_nist
import nltk
from tqdm import tqdm
tqdm.pandas()

def calculate_nist_row(row): 
    ref_tokens = nltk.word_tokenize(row['human_reference'], language='english')
    gen_token = nltk.word_tokenize(row['generation'], language='english')
    score = sentence_nist([ref_tokens], gen_token)
    return score

metric_df["nist_1"] = all_df.progress_apply(calculate_nist_row, axis=1, result_type='expand')



In [ ]:
metric_df.to_csv("en-metric-may2026.csv", index=False)

## Perplexity (Eurollm-1.7b)


In [ ]:
from metric_utils_en import perplexity

In [ ]:
example_text = all_df["generation"].iloc[0]

res = perplexity(
    predictions=[example_text],
    model_id="./models/eurollm-1.7b",
    batch_size=1,  # keep 1 for simplicity
    max_length=1024,
    stride=1024,
    add_start_token=False
)
print(res)

In [ ]:
perplexity_results = perplexity(
    predictions = list(all_df['generation']),
    model_id="../models/eurollm-1.7b",
    max_length=1024,
    stride=1024,
    add_start_token=False)

perplexity_scores = pd.DataFrame(perplexity_results["perplexities"], columns=['perplexity'])

In [ ]:
# append perplexity score to metric_df and save
metric_df["perplexity"] = perplexity_scores


## JS-divergence (based on unigram distribution)

In [ ]:
from collections import Counter
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy
from numpy.linalg import norm

def get_unigram_distribution(text, vocab=None):
    tokens = text.split()
    counter = Counter(tokens)
    if vocab is None:
        vocab = set(counter.keys())
    dist = np.array([counter[word] for word in vocab], dtype=np.float32)
    return dist / dist.sum(), vocab

def calculate_jsd(P, Q):
    _P = P / norm(P, ord=1)
    _Q = Q / norm(Q, ord=1)
    _M = 0.5 * (_P + _Q)
    return 0.5 * (entropy(_P, _M) + entropy(_Q, _M))

def get_jsd_per_example(summary, reference, distribution="unigram"): 
    """
    Compute the Jensen-Shannon Divergence (JSD) between a summary and a reference text
    using unigram word distributions.

    This function tokenizes the input strings, builds unigram frequency distributions 
    over a shared vocabulary, and computes the JSD between the two distributions.

    Parameters:
    ----------
    summary : str
        The generated summary text.

    reference : str
        The reference (gold-standard) text to compare against.

    distribution : str, optional (default="unigram")
        The type of distribution to use. 

    Returns:
    -------
    float
        The Jensen-Shannon Divergence score (between 0 and 1), where lower scores 
        indicate greater similarity between the summary and reference.
    """
    
    # Ensure same vocabulary for both
    if distribution == "unigram": 
        _, vocab = get_unigram_distribution(summary)
        P, _ = get_unigram_distribution(summary, vocab)
        Q, _ = get_unigram_distribution(reference, vocab)
        jsd = calculate_jsd(P, Q)
        
    return jsd


In [ ]:
metric_df['js_divergence'] = all_df.apply(
    lambda row: get_jsd_per_example(row["generation"], row["human_reference"]),
    axis=1
)

# jsd_scores_df = jsd_score.to_frame(name="js_divergence")
# # append jsd scores to csv 
# filtered_df['jsdivergence'] = jsd_score_df
# filtered_df.to_csv("./output/filtered_metric_df.csv", index=False)

## Coverage, Density, Compression (Grusky 2018)
adapted from https://github.com/lil-lab/newsroom/blob/master/newsroom/analyze/fragments.py


Coverage = (Total # of words in extractive fragments) / (Total # of words in the summary)

Compression = (Total # of words in source) / Total # of words in summary

Density = (∑ (length of fragment)^2) / (Total # of words in the summary)

In [ ]:
from fragments_en import Fragments


def compute_fragments_metrics(row):
    summary = row['generation']
    text = (row['patient_file'])
    frag = Fragments(summary, text) 

    coverage = frag.coverage()   
    density = frag.density()     
    compression = frag.compression()  

    return pd.Series({
        'coverage': coverage,
        'density': density,
        'compression': compression
    })

# Apply to each row, axis=1 means row-wise
coverage_density_compression = all_df.progress_apply(compute_fragments_metrics, axis=1)

In [ ]:
metric_df = pd.concat([metric_df, coverage_density_compression], axis=1)

## Redundancy, compression, semantic coherence, abstractivity (Bommasani and Cardie 2020, Koh 2022)

In [ ]:
from metric_utils_en import redundancy, semantic_coherence,abstractivity

# semantic coherence 
semantic_coherence_list = semantic_coherence(all_df, col_name='generation')
semantic_coherence_score = pd.DataFrame(semantic_coherence_list, columns=['semantic_coherence'])

# redundancy
redundancy_score = all_df.progress_apply(redundancy, axis=1)

# abstractivity
abstractivity_score = all_df.progress_apply(lambda row: abstractivity(row, density_df=coverage_density_compression), axis=1)

In [ ]:
metric_df["semantic_coherence"] = semantic_coherence_score
metric_df = pd.concat([metric_df, redundancy_score], axis=1)
metric_df["abstractivity"] = abstractivity_score

## Degree of abstraction: novel n-gram (Tang 2023)

The percentage of novel n-grams signifies the proportion of n-grams in the summary that differ from the original source document.

number of novel n-grams that are not in the source / number of total n-grams in summary

In [ ]:
def novel_ngram_percentage(row, n=1):

    def get_ngrams(text, n):
        words = text.lower().split()
        return set(tuple(words[i:i+n]) for i in range(len(words)-n+1))

    source = row['patient_file']
    summary = row['generation']
    source_ngrams = get_ngrams(source, n)
    summary_ngrams = get_ngrams(summary, n)

    novel_ngrams = summary_ngrams - source_ngrams
    return len(novel_ngrams) / len(summary_ngrams) * 100.0  

metric_df["abstraction_1gram"] = all_df.progress_apply(novel_ngram_percentage, axis=1, n=1).to_frame(name='abstraction_1gram')
metric_df["abstraction_2gram"] = all_df.progress_apply(novel_ngram_percentage, axis=1, n=2).to_frame(name='abstraction_2gram')

# metric_df['abstraction_1gram'], metric_df['abstraction_2gram'] = degree_of_abstraction_1gram, degree_of_abstraction_2gram

## Uniformity (Koh 2022)
"Uniformity measures whether content that are considered important by the reference summary are uniformly scattered across the entire source document. A higher score indicates that important content are scattered across the entire document with no obvious layout bias to take advantage of. This is calculated based on the normalized entropy of the decile positions of salient unigrams in the source text, where salient unigrams are the top 20 keywords extracted1, excluding stopwords, from the reference summary..." (Koh 2022)

Code is based on the above description 

#### Interpretation: 
High uniformity score (≈1) = Important content is scattered across the document.

Low uniformity score (≈0) = Important content is clustered in just a part of the document.

In [ ]:
import numpy as np
import nltk
from nltk.corpus import stopwords
from rake_nltk import Rake
from collections import Counter

nltk.download('stopwords')


def extract_top_keywords(text, top_n=20):
    rake = Rake(language="English")
    rake.extract_keywords_from_text(text)
    ranked_phrases = rake.get_ranked_phrases()  # full phrases

    # split phrases into unigrams
    unigrams = []
    for phrase in ranked_phrases:
        unigrams.extend(phrase.split())

    # take top_n most frequent
    top_keywords = [w for w, _ in Counter(unigrams).most_common(top_n)]
    return top_keywords


def compute_uniformity(row, top_n=20):
    
    # compare patient_file and GPt letter 
    summary = row['generation']
    source_text = row['patient_file']
    # Step 1: extract salient unigrams
    keywords = extract_top_keywords(summary, top_n)

    # Step 2: tokenize source text
    source_tokens = nltk.word_tokenize(source_text.lower())
    total_len = len(source_tokens)

    # Step 3: find positions (index) of keywords in the source 
    source_positions = []
    for i, w in enumerate(source_tokens):
        if w in keywords:
            source_positions.append(i)

    # Step 4: map to deciles (0–9)
    # [0, 0, 2, 5, 5, 9] -> the thrid word appears at 2 decile, the fourth word appear at 5 decile 
    # relative position of the word. e.g., 25/100 = 0.25 = word is at the 25% position in the source
    # scale the relative position into 10 bins by * 10
    deciles = [int((pos / total_len) * 10) for pos in source_positions]
    # if > 10 then it belongs to bin 9 (max)
    deciles = [d if d < 10 else 9 for d in deciles]  

    # Step 5: turns keyword positions into a probability distribution across 10 bins.
    # counts how many keywords in each of the 10 bins.
    counts = np.bincount(deciles, minlength=10)
    probs = counts / counts.sum()

    # Step 6: compute normalized entropy
    entropy = -np.sum([p * np.log(p) for p in probs if p > 0])
    max_entropy = np.log(10)
    normalized_entropy = entropy / max_entropy

    return normalized_entropy


metric_df['uniformity'] = all_df.progress_apply(compute_uniformity, axis=1).to_frame(name='uniformity')

# metric_df['uniformity'] = uniformity_score


## Measure of textual lexical diversity (MTLD) (McCarthy & Jarvis, 2010)
MTLD python library: https://pypi.org/project/taaled/

In [ ]:
from taaled import ld
import spacy
nlp = spacy.load("en_core_web_lg")


def calculate_mtld(row): 
    summary = row['generation']
    doc = nlp(summary)
    # preprocessing: lowercase, remove punct, add suffix "_POS" to each token (e.g., ['there_PRON', 'be_VERB', 'a_DET', 'saying_NOUN'])
    token = [f"{token.lemma_}_{token.pos_}" for token in doc if token.pos_ != "PUNCT" and token.pos_ != "SPACE"]
    ldvals = ld.lexdiv(token)
    return ldvals.mtldo

metric_df['mtld'] = all_df.progress_apply(calculate_mtld, axis=1).to_frame(name='mtld')

# metric_df['mtld'] = mtld_score
    

## Readability (Flesch Reading Ease Index (Kincaid 1975, Lee 2024))

In [ ]:
import spacy
from spacy_syllables import SpacySyllables
from nltk.tokenize import sent_tokenize, word_tokenize
import string
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("syllables", after="tagger")

def flesch_reading_ease(text):
      """
            Flesch Reading Ease score = 206.835 - (1.015 × (Total words / total sentences)) - (84.6 × (total syllables / total words))
      """
      summary = text['generation']
      sentences = sent_tokenize(summary, language='english')
      tokens = word_tokenize(summary, language='english')
      len_tokens_no_punct = len([t for t in tokens if t not in string.punctuation])
      # total number of syllables 
      doc = nlp(summary)
      syllable_parse = [(token._.syllables_count) for token in doc]
      sum_syllable = sum(x for x in syllable_parse if x is not None)
      FRE = 206.835 - float(1.015 * len_tokens_no_punct/len(sentences)) -\
            float(84.6 * sum_syllable/len(tokens))
      return pd.Series({"readability": FRE})

metric_df['readability'] = all_df.progress_apply(flesch_reading_ease, axis=1)



## SARI

In [ ]:
from metric_utils_en import sari

def calculate_sari(row):
    
    sources = [row["patient_file"]]
    predictions = [row["generation"]]
    references = [[row["human_reference"]]]
    sari_score = sari(sources=sources, predictions=predictions, references=references)
    
    return sari_score["sari"]

metric_df["sari"] = all_df.progress_apply(calculate_sari, axis=1)

## Distinct-1, Distinct-2, Distinct-3

- number of distinct unigrams and bigrams divided by total number of generated words
- https://aclanthology.org/N16-1014.pdf

In [ ]:

from collections import Counter
from nltk import ngrams, word_tokenize

def distinct_n(texts, n):
    """
    texts: list of strings (can be one or many documents)
    n: n-gram size (1, 2, or 3)
    returns: distinct-n score
    """
    all_ngrams = []
    sents = sent_tokenize(texts, language = "english")
    for sent in sents:
        tokens = word_tokenize(sent, language = "english")
        all_ngrams += list(ngrams(tokens, n))
    
    if len(all_ngrams) == 0:
        return 0.0
    
    unique_ngrams = set(all_ngrams)
    return len(unique_ngrams) / len(all_ngrams)

metric_df['distinct_1'] = all_df['generation'].progress_apply(lambda x: distinct_n(x, n=1))
metric_df['distinct_2'] = all_df['generation'].progress_apply(lambda x: distinct_n(x, n=2))
metric_df['distinct_3'] = all_df['generation'].progress_apply(lambda x: distinct_n(x, n=3))

## Append the new metrics to the dataframe and save it as csv

In [ ]:
metric_df.to_csv("en-metric-may2026.csv",index=False)